In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)

In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

DATA_CANDIDATES = [
    Path("fake_job_postings.csv"),
    Path("/content/fake_job_postings.csv"),
]

data_path = next(
    (p for p in DATA_CANDIDATES if p.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "Could not find fake_job_postings.csv"
    )

raw_df = pd.read_csv(data_path)

print("Dataset shape:", raw_df.shape)
raw_df.head()

In [ ]:
# ============================================================
# 3. CLEANING & FEATURE ENGINEERING
# ============================================================

df = raw_df.copy()

# Drop columns that were not used in the previous final model
df = df.drop(columns=["salary_range"])

# Preserve the previous location handling
df["location"] = df["location"].fillna(
    "Unknown_location"
)

# Department was explored previously but not used
df = df.drop(columns=["department"])

# ------------------------------------------------------------
# Text-length features
# ------------------------------------------------------------

TEXT_COLUMNS = [
    "title",
    "description",
    "requirements"
]

for col in TEXT_COLUMNS:
    df[f"{col}_len"] = (
        df[col]
        .fillna("")
        .str.len()
    )

# ------------------------------------------------------------
# Combined text
# ------------------------------------------------------------

df["combined_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["requirements"].fillna("")
).str.strip()

print("Cleaned shape:", df.shape)

df[
    [
        "title",
        "description_len",
        "requirements_len",
        "combined_text",
        "fraudulent"
    ]
].head()

In [ ]:
# ============================================================
# 4. STRUCTURED FEATURE DEFINITIONS
# ============================================================

NUM_COLS = [
    "title_len",
    "description_len",
    "requirements_len"
]

CAT_COLS = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
    "location",
]

STRUCTURED_FEATURES = CAT_COLS + NUM_COLS

print("Categorical features:")
print(CAT_COLS)

print("\nNumerical features:")
print(NUM_COLS)

In [ ]:
# ============================================================
# 5. GROUP-AWARE TRAIN / VALIDATION / TEST SPLIT
# ============================================================

groups = df["combined_text"].fillna("")

# ------------------------------------------------------------
# First split:
# Development = 80%
# Test = 20%
# ------------------------------------------------------------

sgkf_test = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

dev_rel_idx, test_rel_idx = next(
    sgkf_test.split(
        df,
        df["fraudulent"],
        groups=groups
    )
)

dev_idx = df.index[dev_rel_idx]
test_idx = df.index[test_rel_idx]

# ------------------------------------------------------------
# Second split:
# Train ≈ 64%
# Validation ≈ 16%
# ------------------------------------------------------------

dev_df = df.loc[dev_idx].copy()

sgkf_val = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_rel_idx, val_rel_idx = next(
    sgkf_val.split(
        dev_df,
        dev_df["fraudulent"],
        groups=dev_df["combined_text"]
    )
)

train_idx = dev_df.index[train_rel_idx]
val_idx = dev_df.index[val_rel_idx]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

In [ ]:
# ============================================================
# 6. VERIFY SPLIT
# ============================================================

y_train = df.loc[
    train_idx, "fraudulent"
].astype(int)

y_val = df.loc[
    val_idx, "fraudulent"
].astype(int)

y_test = df.loc[
    test_idx, "fraudulent"
].astype(int)

print("Fraud rates")
print("-" * 40)

print(
    "Overall:",
    round(df["fraudulent"].mean(), 4)
)

print(
    "Train:",
    round(y_train.mean(), 4)
)

print(
    "Validation:",
    round(y_val.mean(), 4)
)

print(
    "Test:",
    round(y_test.mean(), 4)
)

# Check duplicate-text overlap

train_texts = set(
    df.loc[train_idx, "combined_text"]
)

val_texts = set(
    df.loc[val_idx, "combined_text"]
)

test_texts = set(
    df.loc[test_idx, "combined_text"]
)

print(
    "Train ↔ Validation:",
    len(train_texts & val_texts)
)

print(
    "Train ↔ Test:",
    len(train_texts & test_texts)
)

print(
    "Validation ↔ Test:",
    len(val_texts & test_texts)
)

In [ ]:
# ============================================================
# 7. TF-IDF FEATURES
# ============================================================

X_train_text = df.loc[
    train_idx,
    "combined_text"
].fillna("")

X_val_text = df.loc[
    val_idx,
    "combined_text"
].fillna("")

X_test_text = df.loc[
    test_idx,
    "combined_text"
].fillna("")

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_val_tfidf = tfidf.transform(
    X_val_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

print(
    "Train TF-IDF:",
    X_train_tfidf.shape
)

print(
    "Validation TF-IDF:",
    X_val_tfidf.shape
)

print(
    "Test TF-IDF:",
    X_test_tfidf.shape
)

In [ ]:
# ============================================================
# 8. STRUCTURED FEATURES
# ============================================================

X_struct_train = df.loc[
    train_idx,
    STRUCTURED_FEATURES
]

X_struct_val = df.loc[
    val_idx,
    STRUCTURED_FEATURES
]

X_struct_test = df.loc[
    test_idx,
    STRUCTURED_FEATURES
]

structured_preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            CAT_COLS
        ),
        (
            "num",
            "passthrough",
            NUM_COLS
        ),
    ]
)
X_struct_train_encoded = (
    structured_preprocess.fit_transform(
        X_struct_train
    )
)

X_struct_val_encoded = (
    structured_preprocess.transform(
        X_struct_val
    )
)

X_struct_test_encoded = (
    structured_preprocess.transform(
        X_struct_test
    )
)

In [ ]:
# ============================================================
# 9. FEATURE FUSION
# ============================================================

X_combined_train = hstack([
    X_train_tfidf,
    X_struct_train_encoded
]).tocsr()

X_combined_val = hstack([
    X_val_tfidf,
    X_struct_val_encoded
]).tocsr()

X_combined_test = hstack([
    X_test_tfidf,
    X_struct_test_encoded
]).tocsr()

print(
    "Combined train shape:",
    X_combined_train.shape
)

print(
    "Combined validation shape:",
    X_combined_val.shape
)

print(
    "Combined test shape:",
    X_combined_test.shape
)

In [ ]:
# ============================================================
# 10. TEXT-ONLY BASELINE
# ============================================================

tfidf_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_STATE
)

tfidf_lr.fit(
    X_train_tfidf,
    y_train
)

tfidf_test_prob = (
    tfidf_lr.predict_proba(
        X_test_tfidf
    )[:, 1]
)

tfidf_test_pred = (
    tfidf_test_prob >= 0.5
).astype(int)

In [ ]:
print(
    classification_report(
        y_test,
        tfidf_test_pred,
        target_names=["Real", "Fraud"],
        digits=4
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_test,
            tfidf_test_prob
        ),
        4
    )
)

print(
    "PR-AUC:",
    round(
        average_precision_score(
            y_test,
            tfidf_test_prob
        ),
        4
    )
)

In [ ]:
# ============================================================
# 11. COMBINED TF-IDF + STRUCTURED LOGISTIC REGRESSION
# ============================================================

combined_lr = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_STATE
)

combined_lr.fit(
    X_combined_train,
    y_train
)

print("Combined Logistic Regression trained successfully.")

In [ ]:
# ============================================================
# 12. VALIDATION EVALUATION
# ============================================================

combined_val_prob = combined_lr.predict_proba(
    X_combined_val
)[:, 1]

combined_val_pred = (
    combined_val_prob >= 0.5
).astype(int)

print("COMBINED MODEL — VALIDATION")
print("=" * 50)

print(
    classification_report(
        y_val,
        combined_val_pred,
        target_names=["Real", "Fraud"],
        digits=4
    )
)

combined_val_roc_auc = roc_auc_score(
    y_val,
    combined_val_prob
)

combined_val_pr_auc = average_precision_score(
    y_val,
    combined_val_prob
)

print(
    f"ROC-AUC: {combined_val_roc_auc:.4f}"
)

print(
    f"PR-AUC:  {combined_val_pr_auc:.4f}"
)

In [ ]:
# ============================================================
# 13. TEXT-ONLY VS COMBINED — VALIDATION
# ============================================================

tfidf_val_prob = tfidf_lr.predict_proba(
    X_val_tfidf
)[:, 1]

tfidf_val_pred = (
    tfidf_val_prob >= 0.5
).astype(int)

tfidf_val_f1 = f1_score(
    y_val,
    tfidf_val_pred,
    zero_division=0
)

combined_val_f1 = f1_score(
    y_val,
    combined_val_pred,
    zero_division=0
)

comparison_val = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Structured LR"
    ],
    "Fraud_Precision": [
        precision_score(
            y_val,
            tfidf_val_pred,
            zero_division=0
        ),
        precision_score(
            y_val,
            combined_val_pred,
            zero_division=0
        )
    ],
    "Fraud_Recall": [
        recall_score(
            y_val,
            tfidf_val_pred,
            zero_division=0
        ),
        recall_score(
            y_val,
            combined_val_pred,
            zero_division=0
        )
    ],
    "Fraud_F1": [
        tfidf_val_f1,
        combined_val_f1
    ],
    "ROC_AUC": [
        roc_auc_score(
            y_val,
            tfidf_val_prob
        ),
        combined_val_roc_auc
    ],
    "PR_AUC": [
        average_precision_score(
            y_val,
            tfidf_val_prob
        ),
        combined_val_pr_auc
    ]
})

comparison_val.sort_values(
    "Fraud_F1",
    ascending=False
)

In [ ]:
# ============================================================
# 14. FINAL TEST EVALUATION
# ============================================================

combined_test_prob = combined_lr.predict_proba(
    X_combined_test
)[:, 1]

combined_test_pred = (
    combined_test_prob >= 0.5
).astype(int)

print("COMBINED MODEL — FINAL TEST")
print("=" * 50)

print(
    classification_report(
        y_test,
        combined_test_pred,
        target_names=["Real", "Fraud"],
        digits=4
    )
)

In [ ]:
combined_test_accuracy = accuracy_score(
    y_test,
    combined_test_pred
)

combined_test_precision = precision_score(
    y_test,
    combined_test_pred,
    zero_division=0
)

combined_test_recall = recall_score(
    y_test,
    combined_test_pred,
    zero_division=0
)

combined_test_f1 = f1_score(
    y_test,
    combined_test_pred,
    zero_division=0
)

combined_test_roc_auc = roc_auc_score(
    y_test,
    combined_test_prob
)

combined_test_pr_auc = average_precision_score(
    y_test,
    combined_test_prob
)

print(f"Accuracy:        {combined_test_accuracy:.4f}")
print(f"Fraud Precision: {combined_test_precision:.4f}")
print(f"Fraud Recall:    {combined_test_recall:.4f}")
print(f"Fraud F1:        {combined_test_f1:.4f}")
print(f"ROC-AUC:         {combined_test_roc_auc:.4f}")
print(f"PR-AUC:          {combined_test_pr_auc:.4f}")

In [ ]:
# ============================================================
# 15. FINAL TEXT-ONLY VS COMBINED COMPARISON
# ============================================================

tfidf_test_pred = (
    tfidf_test_prob >= 0.5
).astype(int)

final_comparison = pd.DataFrame({

    "Model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Structured LR"
    ],

    "Accuracy": [
        accuracy_score(
            y_test,
            tfidf_test_pred
        ),
        combined_test_accuracy
    ],

    "Fraud_Precision": [
        precision_score(
            y_test,
            tfidf_test_pred,
            zero_division=0
        ),
        combined_test_precision
    ],

    "Fraud_Recall": [
        recall_score(
            y_test,
            tfidf_test_pred,
            zero_division=0
        ),
        combined_test_recall
    ],

    "Fraud_F1": [
        f1_score(
            y_test,
            tfidf_test_pred,
            zero_division=0
        ),
        combined_test_f1
    ],

    "ROC_AUC": [
        roc_auc_score(
            y_test,
            tfidf_test_prob
        ),
        combined_test_roc_auc
    ],

    "PR_AUC": [
        average_precision_score(
            y_test,
            tfidf_test_prob
        ),
        combined_test_pr_auc
    ]
})

final_comparison.sort_values(
    "Fraud_F1",
    ascending=False
)

In [ ]:
# ============================================================
# 16. F1 COMPARISON
# ============================================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    final_comparison["Model"],
    final_comparison["Fraud_F1"]
)

plt.ylim(0, 1)

plt.ylabel("Fraud F1")
plt.title(
    "Effect of Adding Structured Features to TF-IDF"
)

plt.xticks(
    rotation=15,
    ha="right"
)

# Add values above bars
for bar in bars:
    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.02,
        f"{height:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 17. PR-AUC COMPARISON
# ============================================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    final_comparison["Model"],
    final_comparison["PR_AUC"]
)

plt.ylim(0, 1)

plt.ylabel("PR-AUC")
plt.title(
    "PR-AUC Comparison — Text vs Text + Structured Features"
)

plt.xticks(
    rotation=15,
    ha="right"
)

for bar in bars:
    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.02,
        f"{height:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 18. COMBINED MODEL CONFUSION MATRIX
# ============================================================

cm_combined = confusion_matrix(
    y_test,
    combined_test_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_combined,
    display_labels=["Real", "Fraud"]
)

disp.plot()

plt.title(
    "Confusion Matrix — TF-IDF + Structured LR"
)

plt.show()

In [ ]:
# ============================================================
# 19. IMPROVEMENT OVER TEXT-ONLY BASELINE
# ============================================================

text_f1 = f1_score(
    y_test,
    tfidf_test_pred,
    zero_division=0
)

text_pr_auc = average_precision_score(
    y_test,
    tfidf_test_prob
)

f1_change = combined_test_f1 - text_f1
pr_auc_change = combined_test_pr_auc - text_pr_auc

print(
    f"Change in Fraud F1:  {f1_change:+.4f}"
)

print(
    f"Change in PR-AUC:    {pr_auc_change:+.4f}"
)

In [ ]:
# ============================================================
# THRESHOLD TUNING — COMBINED MODEL
# ============================================================

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

combined_threshold_results = []

for threshold in thresholds:

    val_pred = (
        combined_val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )

    combined_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

combined_threshold_df = pd.DataFrame(
    combined_threshold_results
)

combined_threshold_df.sort_values(
    "f1",
    ascending=False
).head(10)

In [ ]:
best_combined_row = combined_threshold_df.loc[
    combined_threshold_df["f1"].idxmax()
]

best_combined_threshold = (
    best_combined_row["threshold"]
)

print(
    "Best validation threshold:",
    round(best_combined_threshold, 3)
)

print(
    "Validation precision:",
    round(
        best_combined_row["precision"],
        4
    )
)

print(
    "Validation recall:",
    round(
        best_combined_row["recall"],
        4
    )
)

print(
    "Validation F1:",
    round(
        best_combined_row["f1"],
        4
    )
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    combined_threshold_df["threshold"],
    combined_threshold_df["precision"],
    label="Precision"
)

plt.plot(
    combined_threshold_df["threshold"],
    combined_threshold_df["recall"],
    label="Recall"
)

plt.plot(
    combined_threshold_df["threshold"],
    combined_threshold_df["f1"],
    label="F1"
)

plt.axvline(
    best_combined_threshold,
    linestyle="--",
    label=f"Best F1 threshold = {best_combined_threshold:.2f}"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title(
    "Threshold Analysis — TF-IDF + Structured LR"
)

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# ============================================================
# FINAL TEST EVALUATION WITH VALIDATION-SELECTED THRESHOLD
# ============================================================

combined_test_pred_tuned = (
    combined_test_prob >= best_combined_threshold
).astype(int)

print(
    classification_report(
        y_test,
        combined_test_pred_tuned,
        target_names=["Real", "Fraud"],
        digits=4
    )
)

In [ ]:
combined_tuned_precision = precision_score(
    y_test,
    combined_test_pred_tuned,
    zero_division=0
)

combined_tuned_recall = recall_score(
    y_test,
    combined_test_pred_tuned,
    zero_division=0
)

combined_tuned_f1 = f1_score(
    y_test,
    combined_test_pred_tuned,
    zero_division=0
)

print("FINAL COMBINED MODEL")
print("=" * 50)

print(
    f"Threshold:       {best_combined_threshold:.3f}"
)

print(
    f"Precision:       {combined_tuned_precision:.4f}"
)

print(
    f"Recall:          {combined_tuned_recall:.4f}"
)

print(
    f"F1:              {combined_tuned_f1:.4f}"
)

print(
    f"ROC-AUC:         {combined_test_roc_auc:.4f}"
)

print(
    f"PR-AUC:          {combined_test_pr_auc:.4f}"
)

In [ ]:
combined_threshold_comparison = pd.DataFrame({
    "Threshold": [
        0.50,
        best_combined_threshold
    ],

    "Precision": [
        combined_test_precision,
        combined_tuned_precision
    ],

    "Recall": [
        combined_test_recall,
        combined_tuned_recall
    ],

    "F1": [
        combined_test_f1,
        combined_tuned_f1
    ]
})

combined_threshold_comparison

In [ ]:
# ============================================================
# FINAL TEST — VALIDATION-SELECTED THRESHOLD
# ============================================================

best_combined_threshold = 0.72

combined_test_pred_tuned = (
    combined_test_prob >= best_combined_threshold
).astype(int)

print("FINAL TEST — COMBINED MODEL")
print("=" * 50)

print(
    classification_report(
        y_test,
        combined_test_pred_tuned,
        target_names=["Real", "Fraud"],
        digits=4
    )
)
combined_tuned_accuracy = accuracy_score(
    y_test,
    combined_test_pred_tuned
)

combined_tuned_precision = precision_score(
    y_test,
    combined_test_pred_tuned,
    zero_division=0
)

combined_tuned_recall = recall_score(
    y_test,
    combined_test_pred_tuned,
    zero_division=0
)

combined_tuned_f1 = f1_score(
    y_test,
    combined_test_pred_tuned,
    zero_division=0
)

print(f"Accuracy:        {combined_tuned_accuracy:.4f}")
print(f"Fraud Precision: {combined_tuned_precision:.4f}")
print(f"Fraud Recall:    {combined_tuned_recall:.4f}")
print(f"Fraud F1:        {combined_tuned_f1:.4f}")
print(f"ROC-AUC:         {combined_test_roc_auc:.4f}")
print(f"PR-AUC:          {combined_test_pr_auc:.4f}")

In [ ]:
threshold_comparison = pd.DataFrame({
    "Threshold": [0.50, 0.72],

    "Precision": [
        combined_test_precision,
        combined_tuned_precision
    ],

    "Recall": [
        combined_test_recall,
        combined_tuned_recall
    ],

    "F1": [
        combined_test_f1,
        combined_tuned_f1
    ]
})

threshold_comparison

In [ ]:
# If XGBoost is not installed:
# %pip install xgboost

from xgboost import XGBClassifier

X_struct_train_encoded
X_struct_val_encoded
X_struct_test_encoded

y_train
y_val
y_test

In [ ]:
print("Training class distribution:")
print(pd.Series(y_train).value_counts())

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("\nNegative samples:", negative)
print("Positive samples:", positive)
print("scale_pos_weight:", round(scale_pos_weight, 2))

In [ ]:
# ============================================================
# XGBOOST — STRUCTURED FEATURES
# ============================================================

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,

    subsample=0.85,
    colsample_bytree=0.90,

    min_child_weight=5,

    reg_alpha=0.1,
    reg_lambda=5.0,

    objective="binary:logistic",
    eval_metric="logloss",

    scale_pos_weight=scale_pos_weight,

    random_state=RANDOM_STATE,
    n_jobs=-1,

    # Start with CPU for this experiment.
    tree_method="hist"
)

xgb_model.fit(
    X_struct_train_encoded,
    y_train
)

print("XGBoost training complete.")

In [ ]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

xgb_val_prob = xgb_model.predict_proba(
    X_struct_val_encoded
)[:, 1]

xgb_val_pred = (
    xgb_val_prob >= 0.5
).astype(int)

print("XGBoost — Validation")
print("=" * 45)

print(
    classification_report(
        y_val,
        xgb_val_pred,
        target_names=["Real", "Fraud"],
        digits=4
    )
)

xgb_val_roc_auc = roc_auc_score(
    y_val,
    xgb_val_prob
)

xgb_val_pr_auc = average_precision_score(
    y_val,
    xgb_val_prob
)

print(f"ROC-AUC: {xgb_val_roc_auc:.4f}")
print(f"PR-AUC:  {xgb_val_pr_auc:.4f}")

In [ ]:
# ============================================================
# XGBOOST THRESHOLD TUNING
# ============================================================

xgb_thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

xgb_threshold_results = []

for threshold in xgb_thresholds:

    pred = (
        xgb_val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    xgb_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

xgb_threshold_df = pd.DataFrame(
    xgb_threshold_results
)

xgb_threshold_df.sort_values(
    "f1",
    ascending=False
).head(10)

In [ ]:
best_xgb_row = xgb_threshold_df.loc[
    xgb_threshold_df["f1"].idxmax()
]

best_xgb_threshold = (
    best_xgb_row["threshold"]
)

print(
    "Best XGBoost threshold:",
    round(best_xgb_threshold, 2)
)

print(
    "Validation Precision:",
    round(best_xgb_row["precision"], 4)
)

print(
    "Validation Recall:",
    round(best_xgb_row["recall"], 4)
)

print(
    "Validation F1:",
    round(best_xgb_row["f1"], 4)
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    xgb_threshold_df["threshold"],
    xgb_threshold_df["precision"],
    label="Precision"
)

plt.plot(
    xgb_threshold_df["threshold"],
    xgb_threshold_df["recall"],
    label="Recall"
)

plt.plot(
    xgb_threshold_df["threshold"],
    xgb_threshold_df["f1"],
    label="F1"
)

plt.axvline(
    best_xgb_threshold,
    linestyle="--",
    label=f"Best threshold = {best_xgb_threshold:.2f}"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("XGBoost — Threshold Analysis")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

xgb_test_prob = xgb_model.predict_proba(
    X_struct_test_encoded
)[:, 1]

xgb_test_pred = (
    xgb_test_prob >= best_xgb_threshold
).astype(int)

print("XGBoost — FINAL TEST")
print("=" * 50)

print(
    classification_report(
        y_test,
        xgb_test_pred,
        target_names=["Real", "Fraud"],
        digits=4
    )
)

In [ ]:
xgb_test_accuracy = accuracy_score(
    y_test,
    xgb_test_pred
)

xgb_test_precision = precision_score(
    y_test,
    xgb_test_pred,
    zero_division=0
)

xgb_test_recall = recall_score(
    y_test,
    xgb_test_pred,
    zero_division=0
)

xgb_test_f1 = f1_score(
    y_test,
    xgb_test_pred,
    zero_division=0
)

xgb_test_roc_auc = roc_auc_score(
    y_test,
    xgb_test_prob
)

xgb_test_pr_auc = average_precision_score(
    y_test,
    xgb_test_prob
)

print(f"Threshold:       {best_xgb_threshold:.2f}")
print(f"Accuracy:        {xgb_test_accuracy:.4f}")
print(f"Fraud Precision: {xgb_test_precision:.4f}")
print(f"Fraud Recall:    {xgb_test_recall:.4f}")
print(f"Fraud F1:        {xgb_test_f1:.4f}")
print(f"ROC-AUC:         {xgb_test_roc_auc:.4f}")
print(f"PR-AUC:          {xgb_test_pr_auc:.4f}")

In [ ]:
xgb_comparison = pd.DataFrame({
    "Model": [
        "Structured XGBoost"
    ],

    "Fraud_F1": [
        xgb_test_f1
    ],

    "ROC_AUC": [
        xgb_test_roc_auc
    ],

    "PR_AUC": [
        xgb_test_pr_auc
    ]
})

xgb_comparison


In [ ]:
# ============================================================
# XGBOOST FEATURE IMPORTANCE
# ============================================================

ohe = structured_preprocess.named_transformers_["cat"]

cat_feature_names = (
    ohe.get_feature_names_out(CAT_COLS)
)

structured_feature_names = np.concatenate([
    cat_feature_names,
    NUM_COLS
])

importance_df = pd.DataFrame({
    "Feature": structured_feature_names,
    "Importance": xgb_model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(20)
)

importance_df

In [ ]:
plt.figure(figsize=(9, 7))

plot_df = importance_df.sort_values(
    "Importance",
    ascending=True
)

plt.barh(
    plot_df["Feature"],
    plot_df["Importance"]
)

plt.xlabel("XGBoost Feature Importance")
plt.ylabel("Feature")
plt.title(
    "Top Structured Features — XGBoost"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from itertools import product

In [ ]:
# Current / baseline TF-IDF configuration
X_train = df.loc[train_idx, "combined_text"].fillna("")
X_val = df.loc[val_idx, "combined_text"].fillna("")

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print("Train shape:", X_train_tfidf.shape)
print("Validation shape:", X_val_tfidf.shape)

In [ ]:
results_lr = []

C_values = [0.1, 0.3, 0.5, 1, 2, 5, 10]
class_weights = [None, "balanced"]

for C, class_weight in product(C_values, class_weights):

    model = LogisticRegression(
        C=C,
        class_weight=class_weight,
        max_iter=2000,
        solver="liblinear",
        random_state=42
    )

    model.fit(X_train_tfidf, y_train)

    y_prob = model.predict_proba(X_val_tfidf)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    fraud_f1 = f1_score(y_val, y_pred, pos_label=1)

    results_lr.append({
        "C": C,
        "class_weight": class_weight,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "Fraud_F1": fraud_f1,
        "ROC_AUC": roc_auc_score(y_val, y_prob)
    })

lr_results_df = pd.DataFrame(results_lr)

lr_results_df = lr_results_df.sort_values(
    by="Fraud_F1",
    ascending=False
)

print(lr_results_df)

In [ ]:
best_lr_config = lr_results_df.iloc[0]

print("\nBest Logistic Regression Configuration:")
print(best_lr_config)

In [ ]:
best_C = best_lr_config["C"]
best_class_weight = best_lr_config["class_weight"]
tfidf_configs = [
    {
        "ngram_range": (1, 1),
        "min_df": 1,
        "max_df": 0.9,
        "sublinear_tf": True
    },
    {
        "ngram_range": (1, 1),
        "min_df": 2,
        "max_df": 0.9,
        "sublinear_tf": True
    },
    {
        "ngram_range": (1, 1),
        "min_df": 3,
        "max_df": 0.9,
        "sublinear_tf": True
    },
    {
        "ngram_range": (1, 2),
        "min_df": 1,
        "max_df": 0.9,
        "sublinear_tf": True
    },
    {
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.9,
        "sublinear_tf": True
    },
    {
        "ngram_range": (1, 2),
        "min_df": 3,
        "max_df": 0.9,
        "sublinear_tf": True
    }
]

In [ ]:
results_tfidf = []

for config in tfidf_configs:

    print("\nTesting:", config)

    vectorizer = TfidfVectorizer(
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
        max_df=config["max_df"],
        sublinear_tf=config["sublinear_tf"]
    )

    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)

    model = LogisticRegression(
        C=best_C,
        class_weight=best_class_weight,
        max_iter=2000,
        solver="liblinear",
        random_state=42
    )

    model.fit(X_train_vec, y_train)

    y_prob = model.predict_proba(X_val_vec)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    results_tfidf.append({
        "ngram_range": config["ngram_range"],
        "min_df": config["min_df"],
        "max_df": config["max_df"],
        "sublinear_tf": config["sublinear_tf"],
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "Fraud_F1": f1_score(y_val, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_val, y_prob)
    })

tfidf_results_df = pd.DataFrame(results_tfidf)

tfidf_results_df = tfidf_results_df.sort_values(
    by="Fraud_F1",
    ascending=False
)

print("\nTF-IDF Tuning Results:")
print(tfidf_results_df)

In [ ]:
best_tfidf_config = tfidf_results_df.iloc[0]

print("\nBest TF-IDF Configuration:")
print(best_tfidf_config)
final_vectorizer = TfidfVectorizer(
    ngram_range=best_tfidf_config["ngram_range"],
    min_df=int(best_tfidf_config["min_df"]),
    max_df=float(best_tfidf_config["max_df"]),
    sublinear_tf=best_tfidf_config["sublinear_tf"]
)

X_train_final = final_vectorizer.fit_transform(X_train)
X_val_final = final_vectorizer.transform(X_val)

final_model = LogisticRegression(
    C=best_C,
    class_weight=best_class_weight,
    max_iter=2000,
    solver="liblinear",
    random_state=42
)

final_model.fit(X_train_final, y_train)

y_prob = final_model.predict_proba(X_val_final)[:, 1]

In [ ]:
threshold_results = []

thresholds = np.arange(0.10, 0.91, 0.01)

for threshold in thresholds:

    y_pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "Fraud_F1": f1_score(
            y_val,
            y_pred,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df.sort_values(
    by="Fraud_F1",
    ascending=False
)

print(threshold_df.head(10))

best_threshold = threshold_df.iloc[0]["Threshold"]

print("\nBest Threshold:", best_threshold)

In [ ]:
final_pred = (y_prob >= best_threshold).astype(int)

final_results = {
    "Accuracy": accuracy_score(y_val, final_pred),
    "Precision": precision_score(
        y_val,
        final_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val,
        final_pred,
        zero_division=0
    ),
    "Fraud F1": f1_score(
        y_val,
        final_pred,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val,
        y_prob
    )
}

print("\n===== FINAL TUNED TF-IDF + LOGISTIC REGRESSION =====")

for metric, value in final_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
threshold_results = []

thresholds = np.arange(0.10, 0.91, 0.01)

for threshold in thresholds:
    
    y_pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "Fraud_F1": f1_score(y_val, y_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df.sort_values(
    by="Fraud_F1",
    ascending=False
)

print(threshold_df.head(15))

In [ ]:
TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=...,          # best value from your TF-IDF tuning
    max_df=0.9,
    sublinear_tf=True
)

LogisticRegression(
    C=10,
    class_weight="balanced",
    max_iter=2000,
    solver="liblinear",
    random_state=42
)

threshold = 0.49

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
X_train = X_train.fillna("").astype(str)
X_val = X_val.fillna("").astype(str)

y_train = y_train.astype(int)
y_val = y_val.astype(int)

print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True))

In [ ]:
train_df = pd.DataFrame({
    "text": X_train.values,
    "label": y_train.values
})

val_df = pd.DataFrame({
    "text": X_val.values,
    "label": y_val.values
})

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

print(train_dataset)
print(val_dataset)

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
train_lengths = [
    len(tokenizer(text, truncation=False)["input_ids"])
    for text in X_train
]

print("Maximum length:", max(train_lengths))
print("95th percentile:", np.percentile(train_lengths, 95))
print("99th percentile:", np.percentile(train_lengths, 99))

In [ ]:
MAX_LENGTH = 256

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=1
    ).numpy()[:, 1]

    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),

        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),

        "fraud_f1": f1_score(
            labels,
            predictions,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            labels,
            probabilities
        )
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_fake_jobs",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="fraud_f1",
    greater_is_better=True,

    logging_steps=50,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=42
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print("\n===== DISTILBERT VALIDATION RESULTS =====")

for key, value in results.items():
    print(f"{key}: {value}")

In [ ]:
predictions = trainer.predict(val_dataset)

logits = predictions.predictions

probabilities = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()[:, 1]

In [ ]:
threshold_results = []

thresholds = np.arange(0.10, 0.91, 0.01)

for threshold in thresholds:

    y_pred = (
        probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),

        "Accuracy": accuracy_score(
            y_val,
            y_pred
        ),

        "Precision": precision_score(
            y_val,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_val,
            y_pred,
            zero_division=0
        ),

        "Fraud_F1": f1_score(
            y_val,
            y_pred,
            zero_division=0
        ),

        "ROC_AUC": roc_auc_score(
            y_val,
            probabilities
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df.sort_values(
    by="Fraud_F1",
    ascending=False
)

print(threshold_df.head(15))

In [ ]:
best_threshold = threshold_df.iloc[0]["Threshold"]

final_pred = (
    probabilities >= best_threshold
).astype(int)

print("\n===== FINAL TUNED DISTILBERT =====")

print("Best Threshold:", best_threshold)

print(
    "Accuracy:",
    round(
        accuracy_score(y_val, final_pred),
        4
    )
)

print(
    "Precision:",
    round(
        precision_score(
            y_val,
            final_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_val,
            final_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "Fraud F1:",
    round(
        f1_score(
            y_val,
            final_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_val,
            probabilities
        ),
        4
    )
)

In [ ]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(y_train)
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float
)

print("Class weights:", class_weights)

In [ ]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        weights = class_weights.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_fct(
            logits,
            labels
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import AutoModelForSequenceClassification

model_weighted = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
training_args_weighted = TrainingArguments(
    output_dir="./distilbert_weighted",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=4,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="fraud_f1",
    greater_is_better=True,

    logging_steps=50,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=42
)

In [ ]:
weighted_trainer = WeightedTrainer(
    model=model_weighted,
    args=training_args_weighted,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

weighted_trainer.train()

In [ ]:
results = weighted_trainer.evaluate()

print("\n===== WEIGHTED DISTILBERT EVALUATION =====")

for key, value in results.items():
    print(f"{key}: {value}")

In [ ]:
predictions = weighted_trainer.predict(val_dataset)

logits = predictions.predictions

probabilities = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()[:, 1]

print("Number of predictions:", len(probabilities))

In [ ]:
threshold_results = []

thresholds = np.arange(0.10, 0.91, 0.01)

for threshold in thresholds:

    y_pred = (probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),

        "Accuracy": accuracy_score(
            y_val,
            y_pred
        ),

        "Precision": precision_score(
            y_val,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_val,
            y_pred,
            zero_division=0
        ),

        "Fraud_F1": f1_score(
            y_val,
            y_pred,
            zero_division=0
        ),

        "ROC_AUC": roc_auc_score(
            y_val,
            probabilities
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df.sort_values(
    by="Fraud_F1",
    ascending=False
).reset_index(drop=True)

print("\n===== TOP 15 THRESHOLDS =====")

print(threshold_df.head(15))

In [ ]:
best_threshold = threshold_df.iloc[0]["Threshold"]

print("\nBest Threshold:", best_threshold)

In [ ]:
final_pred = (
    probabilities >= best_threshold
).astype(int)

final_accuracy = accuracy_score(y_val, final_pred)

final_precision = precision_score(
    y_val,
    final_pred,
    zero_division=0
)

final_recall = recall_score(
    y_val,
    final_pred,
    zero_division=0
)

final_f1 = f1_score(
    y_val,
    final_pred,
    zero_division=0
)

final_auc = roc_auc_score(
    y_val,
    probabilities
)

print("\n===== FINAL WEIGHTED DISTILBERT RESULTS =====")

print(f"Best Threshold: {best_threshold:.2f}")
print(f"Accuracy:       {final_accuracy:.4f}")
print(f"Precision:      {final_precision:.4f}")
print(f"Recall:         {final_recall:.4f}")
print(f"Fraud F1:       {final_f1:.4f}")
print(f"ROC-AUC:        {final_auc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_val, final_pred)

print("\n===== CONFUSION MATRIX =====")
print(cm)

print("\n===== CLASSIFICATION REPORT =====")

print(
    classification_report(
        y_val,
        final_pred,
        target_names=["Real Job", "Fake Job"],
        digits=4
    )
)

In [ ]:
# Use the config you already found best on validation — no more searching
final_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=1, max_df=0.9, sublinear_tf=True
)
X_train_final = final_vectorizer.fit_transform(X_train_text)
X_test_final = final_vectorizer.transform(X_test_text)

final_lr = LogisticRegression(
    C=10, class_weight="balanced", max_iter=2000,
    solver="liblinear", random_state=RANDOM_STATE
)
final_lr.fit(X_train_final, y_train)

test_prob = final_lr.predict_proba(X_test_final)[:, 1]

print("=== TEST SET, threshold=0.50 (default) ===")
test_pred_default = (test_prob >= 0.5).astype(int)
print(classification_report(y_test, test_pred_default, target_names=["Real", "Fraud"], digits=4))
print("ROC-AUC:", roc_auc_score(y_test, test_prob))
print("PR-AUC :", average_precision_score(y_test, test_prob))

print("\n=== TEST SET, threshold=0.49 (chosen on validation, applied once) ===")
test_pred_tuned = (test_prob >= 0.49).astype(int)
print(classification_report(y_test, test_pred_tuned, target_names=["Real", "Fraud"], digits=4))

In [ ]:
from transformers import EarlyStoppingCallback

MAX_LENGTH = 512  # was 256 — was truncating most postings badly

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="./distilbert_fake_jobs_v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,     # smaller batch, seq len doubled
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,     # keeps effective batch size similar to before
    num_train_epochs=8,                # let early stopping cut it off
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="fraud_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

# Unweighted — your own results showed weighted loss hurt F1 (0.713 vs 0.733)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
val_results = trainer.evaluate()
print(val_results)

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

structured_preprocess_scaled = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ("num", StandardScaler(), NUM_COLS),
])

svm_pipeline = Pipeline([
    ("prep", structured_preprocess_scaled),
    ("svm", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE)),
])

svm_pipeline.fit(df.loc[train_idx, CAT_COLS + NUM_COLS], y_train)
svm_pred = svm_pipeline.predict(df.loc[test_idx, CAT_COLS + NUM_COLS])
svm_prob = svm_pipeline.predict_proba(df.loc[test_idx, CAT_COLS + NUM_COLS])[:, 1]

print(classification_report(y_test, svm_pred, target_names=["Real", "Fraud"], digits=4))
print("ROC-AUC:", roc_auc_score(y_test, svm_prob))

In [ ]:
import re
from scipy.sparse import hstack, csr_matrix

def add_fraud_signal_features(frame, raw_frame):
    frame = frame.copy()
    frame["company_profile_missing"] = frame["company_profile"].isna().astype(int)
    frame["salary_missing"] = raw_frame["salary_range"].isna().astype(int)  # grab before you dropped it
    text = frame["combined_text"].fillna("")
    frame["has_email"] = text.str.contains(r"[\w\.-]+@[\w\.-]+", regex=True).astype(int)
    frame["has_url"] = text.str.contains(r"http[s]?://|www\.", regex=True).astype(int)
    frame["exclaim_count"] = text.str.count("!")
    frame["dollar_sign_count"] = text.str.count(r"\$")
    frame["caps_word_ratio"] = text.apply(
        lambda t: sum(1 for w in t.split() if w.isupper() and len(w) > 1) / max(len(t.split()), 1)
    )
    return frame

df = add_fraud_signal_features(df, raw_df)  # raw_df = your original unmodified load

NEW_NUM_COLS = NUM_COLS + [
    "company_profile_missing", "salary_missing", "has_email",
    "has_url", "exclaim_count", "dollar_sign_count", "caps_word_ratio",
]

scaler = StandardScaler()
X_struct_train_scaled = scaler.fit_transform(df.loc[train_idx, NEW_NUM_COLS])
X_struct_test_scaled = scaler.transform(df.loc[test_idx, NEW_NUM_COLS])

X_fusion_train = hstack([X_train_final, csr_matrix(X_struct_train_scaled)])
X_fusion_test = hstack([X_test_final, csr_matrix(X_struct_test_scaled)])

fusion_lr = LogisticRegression(C=10, class_weight="balanced", max_iter=2000, solver="liblinear", random_state=RANDOM_STATE)
fusion_lr.fit(X_fusion_train, y_train)

fusion_prob = fusion_lr.predict_proba(X_fusion_test)[:, 1]
fusion_pred = (fusion_prob >= 0.5).astype(int)

print(classification_report(y_test, fusion_pred, target_names=["Real", "Fraud"], digits=4))
print("ROC-AUC:", roc_auc_score(y_test, fusion_prob))
print("PR-AUC :", average_precision_score(y_test, fusion_prob))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

groups_all = df["combined_text"].fillna("")
X_all_text = df["combined_text"].fillna("")
y_all = df["fraudulent"].astype(int)

skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_f1, cv_roc = [], []

for fold, (tr_idx, te_idx) in enumerate(skf.split(df, y_all, groups=groups_all)):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.9, sublinear_tf=True)
    X_tr = vec.fit_transform(X_all_text.iloc[tr_idx])
    X_te = vec.transform(X_all_text.iloc[te_idx])

    clf = LogisticRegression(C=10, class_weight="balanced", max_iter=2000, solver="liblinear", random_state=RANDOM_STATE)
    clf.fit(X_tr, y_all.iloc[tr_idx])

    prob = clf.predict_proba(X_te)[:, 1]
    pred = (prob >= 0.5).astype(int)

    fold_f1 = f1_score(y_all.iloc[te_idx], pred)
    fold_roc = roc_auc_score(y_all.iloc[te_idx], prob)
    cv_f1.append(fold_f1)
    cv_roc.append(fold_roc)
    print(f"Fold {fold+1}: F1={fold_f1:.4f}, ROC-AUC={fold_roc:.4f}")

print(f"\nMean Fraud F1: {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")
print(f"Mean ROC-AUC:  {np.mean(cv_roc):.4f} ± {np.std(cv_roc):.4f}")

In [ ]:
test_dataset = Dataset.from_pandas(
    pd.DataFrame({"text": X_test_text.values, "label": y_test.values}),
    preserve_index=False
)
test_dataset = test_dataset.map(tokenize_function, batched=True)

test_predictions = trainer.predict(test_dataset)
test_probs = torch.softmax(torch.tensor(test_predictions.predictions), dim=1).numpy()[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

print(classification_report(y_test, test_preds, target_names=["Real", "Fraud"], digits=4))
print("ROC-AUC:", roc_auc_score(y_test, test_probs))
print("PR-AUC :", average_precision_score(y_test, test_probs))